## DATASET 1

This notebook loads dataset from MITRE and prepares our first dataset for the project

- using MITRE dataset as large database we going to search from (the **retrieval corpus**)
- generates `mitre_techniques.csv`

In [2]:
#MITRE's recommended approach to loading the dataset
from stix2 import MemoryStore, Filter
import pandas as pd

FILE_PATH = "enterprise-attack-19.2.json"

mitre_data = MemoryStore()
mitre_data.load_from_file(FILE_PATH)

print("MITRE ATT&CK dataset loaded!")

MITRE ATT&CK dataset loaded!


In [3]:
all_objects = mitre_data.query()

print("Total objects:", len(all_objects))

Total objects: 26086


In [4]:
from collections import Counter

object_types = Counter(obj["type"] for obj in all_objects)

for obj_type, count in object_types.most_common():
    print(f"{obj_type}: {count}")

relationship: 21262
x-mitre-analytic: 1758
attack-pattern: 858
malware: 733
x-mitre-detection-strategy: 699
course-of-action: 268
intrusion-set: 191
x-mitre-data-component: 109
tool: 95
campaign: 56
x-mitre-data-source: 38
x-mitre-tactic: 15
x-mitre-collection: 1
x-mitre-matrix: 1
identity: 1
marking-definition: 1


**MITRE specifically documents techniques and sub-techniques as attack-pattern objects**

In [5]:
techniques = mitre_data.query([
    Filter("type", "=", "attack-pattern")
])

print("Attack patterns found:", len(techniques))

Attack patterns found: 858


In [6]:
#filter queries
def is_active(obj):
    return (
        not obj.get("revoked", False)
        and not obj.get("x_mitre_deprecated", False)
    )

techniques = [t for t in techniques if is_active(t)]

print("Active techniques/sub-techniques:", len(techniques))

Active techniques/sub-techniques: 697


for the project, need:

- ATT&CK ID
- Name
- Description
- Technique/Sub-technique
- Platforms
- Tactics
- STIX ID

In [7]:
def get_attack_id(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")
    return None

In [8]:
technique_rows = []

for technique in techniques:

    attack_id = get_attack_id(technique)

    tactics = [
        phase["phase_name"]
        for phase in technique.get("kill_chain_phases", [])
        if phase.get("kill_chain_name") == "mitre-attack"
    ]

    technique_rows.append({
        "technique_id": attack_id,
        "name": technique.get("name"),
        "description": technique.get("description", ""),
        "is_subtechnique": technique.get(
            "x_mitre_is_subtechnique",
            False
        ),
        "platforms": technique.get(
            "x_mitre_platforms",
            []
        ),
        "tactics": tactics,
        "stix_id": technique.get("id")
    })

techniques_df = pd.DataFrame(technique_rows)

In [9]:
techniques_df.head()

,technique_id,name,description,is_subtechnique,platforms,tactics,stix_id
0,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into pro...,True,[Windows],"[stealth, privilege-escalation]",attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...
1,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...,True,[Windows],"[execution, persistence, privilege-escalation]",attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...
2,T1205.002,Socket Filters,Adversaries may attach filters to a network so...,True,"[Linux, macOS, Windows]","[stealth, persistence, command-and-control]",attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...
3,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...,True,"[Linux, macOS, Windows]",[collection],attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...
4,T1021.005,VNC,Adversaries may use [Valid Accounts](https://a...,True,"[Linux, Windows, macOS]",[lateral-movement],attack-pattern--01327cde-66c4-4123-bf34-5f258d...


In [10]:
print(techniques_df.shape)
print()

print(
    techniques_df["is_subtechnique"]
    .value_counts()
)

(697, 7)

is_subtechnique
True     475
False    222
Name: count, dtype: int64


In [11]:
#check missing values
print(
    techniques_df[
        ["technique_id", "name", "description"]
    ].isnull().sum()
)

technique_id    0
name            0
description     0
dtype: int64


In [12]:
#check duplicates
print(
    "Duplicate ATT&CK IDs:",
    techniques_df["technique_id"].duplicated().sum()
)

Duplicate ATT&CK IDs: 0


In [13]:
techniques_df.to_csv(
    "mitre_techniques.csv",
    index=False
)
print ("saved!")

saved!
